# Qwen y vLLM: preparación · 2.1.4

Declarar artefacto/servidor no arranca GPU ni descarga pesos. [Preparar WSL](../../docs/VLLM_GPU_214.md). Se registra pendiente si falta el endpoint.

In [2]:
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'


In [3]:
artifact = toolkit.model_artifact('Qwen/Qwen3-0.6B', revision='main', metadata={'scope': 'declaration only'})
server = toolkit.model_server(artifact, max_model_len=2048, max_num_seqs=1, gpu_memory_utilization=0.75)
print(artifact.inspect())
print(server.inspect())


{'model': 'Qwen/Qwen3-0.6B', 'base_model': None, 'adapter_path': None, 'tokenizer': None, 'revision': 'main', 'quantization': None, 'metadata': {'scope': 'declaration only'}}
{'backend': 'vllm', 'installed': False, 'command': ['vllm', 'serve', 'Qwen/Qwen3-0.6B', '--host', '127.0.0.1', '--port', '8000', '--served-model-name', 'Qwen/Qwen3-0.6B', '--gpu-memory-utilization', '0.75', '--max-model-len', '2048', '--max-num-seqs', '1', '--enable-auto-tool-choice', '--tool-call-parser', 'hermes', '--generation-config', 'vllm', '--api-key', '[REDACTED]'], 'endpoint': {'url': 'http://127.0.0.1:8000/v1', 'model': 'Qwen/Qwen3-0.6B', 'backend': 'vllm', 'api_key_configured': '[REDACTED]', 'pid': None, 'owned': False}, 'configuration': {'schema_version': 'agentic_systems.serving.v1', 'backend': 'vllm', 'artifact': {'schema_version': 'agentic_systems.serving.v1', 'adapter_path': None, 'revision': 'main', 'quantization': None, 'metadata': {'scope': 'declaration only'}, 'model': 'Qwen/Qwen3-0.6B', 'base_

In [ ]:
@toolkit.tool
def certify(value: int) -> dict:
    return {'value': value, 'verified': True}
capability = toolkit.skill(name='evidence', tools=(certify,))
actor_framework = 'native'
runtime = toolkit.runtime(provider='vllm-runtime', model='Qwen/Qwen3-0.6B', endpoint='http://127.0.0.1:8000/v1', api_key='vllm', max_tokens=512, max_retries=0)
actor = toolkit.Agent('verification', runtime=runtime, framework=toolkit.framework(actor_framework), skills=(capability,), instructions='Call certify once with the supplied value. The Tool output is the result.', contract={'tool_expectation': {'all_of': ['certify']}, 'completion': 'when_required_tools_satisfied'}, policy={'repair': False})
result = actor.run({'value': 37}, timeout_s=120)
events = [event for node in result.walk() for event in node.tool_events]
assert result.ok and result.check_invariants().ok, result.to_dict()
assert any((e.name == 'certify' and e.ok and (e.input == {'value': 37}) for e in events))
print({'input': {'value': 37}, 'process': 'runtime + actor -> Tool event', 'output': result.data})
